In [ ]:
import pandas as pd
import numpy as np

# 1) Charger les données
price_resid = pd.read_csv("../data/deseasonalised/price_resid.csv", index_col=0)
price_res = price_resid["price_deseasoned"].copy()

# si besoin
price_res.index = pd.to_datetime(price_res.index)

# 2) Détection itérative des spikes
is_spike = pd.Series(False, index=price_res.index)

while True:
    sigma = price_res[~is_spike].std()
    new_spikes = (price_res.abs() > 3 * sigma) & (~is_spike)

    if new_spikes.sum() == 0:
        break

    is_spike = is_spike | new_spikes

# 3) Créer une nouvelle série filtrée
price_res_filtered = price_res.copy()

for t in price_res_filtered.index[is_spike]:
    same_context = (
        (price_res_filtered.index.dayofweek == t.dayofweek) &
        (price_res_filtered.index.hour == t.hour) &
        (~is_spike)
    )

    replacement_values = price_res[same_context]

    # fallback simple si on ne trouve rien
    if len(replacement_values) == 0:
        replacement_values = price_res[~is_spike]

    price_res_filtered.loc[t] = replacement_values.mean()

# 4) Créer une nouvelle base de données
price_resid_filtered = price_resid.copy()
price_resid_filtered["price_deseasoned_filtered"] = price_res_filtered
price_resid_filtered["is_spike"] = is_spike

# 5) Sauvegarder
price_resid_filtered.to_csv("../data/deseasonalised/price_resid_filtered.csv")

print("Nombre de spikes détectés :", is_spike.sum())
print("Fichier sauvegardé : ../data/deseasonalised/price_resid_filtered.csv")

In [ ]:
import matplotlib.pyplot as plt

# afficher toute la série filtrée
plt.figure(figsize=(12,5))
plt.plot(price_res_filtered, label="résidus filtrés")
plt.title("Résidus filtrés")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(price_res, alpha=0.5, label="original")
plt.plot(price_res_filtered, label="filtré")
plt.title("Comparaison résidus originaux vs filtrés")
plt.legend()
plt.show()